In [ ]:
from lib2to3.fixes.fix_input import context

from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.graph import START, END, MessagesState, StateGraph, add_messages
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated, List, Literal, TypedDict
from langchain_core.tools import tool

from dotenv import load_dotenv
from pprint import pprint

from tools.vector_store_ops_new import ensure_vectorstore, retrieve_context
# from tools.vectorstore_ops import ensure_vectorstore, retrieve_context, generate_embeddings

load_dotenv()

# Define State

In [ ]:
class State(TypedDict):
    messages: Annotated[List, add_messages]

# Define Tools

In [ ]:
@tool
def get_stock_price(symbol: str) -> float:
    """Return the current price of a stock given the stock symbol
    :param symbol: stock symbol
    :return: current price of the stock"""
    return {"MSFT": 200.3, "AAPL": 100.4, "AMZN": 150.0, "RIL": 87.6}.get(symbol, 0.0)


def search_vectorstore(prompt: str) -> List:
    """Search vector store for any documentation or info on the following topics:
        - general linux administration
        - redhat linux administration
        - suse linux administration
        - dancing dinosaur project
        - botteicher key
    :param prompt: topic to search for
    :return: list of documents from vector store
    """
    vs = ensure_vectorstore("documentation")
    # generate_embeddings(vs)
    context = retrieve_context(vs, prompt)
    return context


tools = [get_stock_price, search_vectorstore]

# Initialize llm and bind tools

In [ ]:
llm = ChatOllama(model="llama3.1:8b-instruct-q4_k_m", num_ctx=4096)
llm_with_tools = llm.bind_tools(tools)

# Setup Memory

In [ ]:
memory = MemorySaver()

# Setup Graph and create nodes, edges

In [ ]:
def chatbot(state: State) -> MessagesState:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)

builder.add_node(chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")

graph = builder.compile(checkpointer=memory)

from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}

prompt = "I want to buy 20 AMZN stocks using current price. Then 15 MSFT. What will be the total cost in Indian Rupees? Output only the total cost"
state = graph.invoke(
    {"messages": [{"role": "user", "content": prompt}]}, config=config1
)
print(state["messages"][-1].content)
pprint(state)